In [1]:
from __future__ import annotations
import logging
import hashlib
import os

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal
from uuid import uuid4

from dotenv import load_dotenv
from pydantic import BaseModel, Field



In [2]:
## resolve project root directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")


False

In [3]:
PROJECT_ROOT

PosixPath('/home/thimu/github_vs/protoRAG/rag-pipeline')

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)-7s | %(name)s | %(message)s",
)
log = logging.getLogger("rag")

In [5]:
def _path(env_key: str, default: str) -> Path:
    """Resolve env-provided paths relative to project root unless absolute."""
    p = Path(os.getenv(env_key, default))
    return p.resolve() if p.is_absolute() else (PROJECT_ROOT / p).resolve()

In [6]:
class Config:
    # ollama
    OLLAMA_HOST: str = os.getenv("OLLAMA_HOST", "localhost")
    OLLAMA_PORT: int = int(os.getenv("OLLAMA_PORT", 11434))
    OLLAMA_MODEL: str = os.getenv("OLLAMA_MODEL", "gemma-4-e4b:latest")
    EMBEDDING_MODEL: str = os.getenv("EMBEDDING_MODEL", "embeddinggemma:latest")

    @property
    def OLLAMA_BASE_URL(self) -> str:
        return f"http://{self.OLLAMA_HOST}:{self.OLLAMA_PORT}"
    
    # Generation
    LLM_TEMPERATURE: float = float(os.getenv("LLM_TEMPERATURE", 0.0))
    LLM_NUM_CTX: int = int(os.getenv("LLM_NUM_CTX", 8192))

    # Retrieval
    CHUNK_SIZE: int = int(os.getenv("CHUNK_SIZE", 800))
    CHUNK_OVERLAP: int = int(os.getenv("CHUNK_OVERLAP", 120))
    TOP_K: int = int(os.getenv("TOP_K", 5))

    # Paths:
    CHROMA_PERSIST_DIR: Path = _path("CHROMA_PERSIST_DIR", "chroma_db")
    DATA_RAW_DIR: Path = _path("DATA_RAW_DIR", "data/raw")
    DATA_PROCESSED_DIR: Path = _path("DATA_PROCESSED_DIR", "data/processed")

cfg = Config()


In [7]:
cfg.CHROMA_PERSIST_DIR.mkdir(parents=True, exist_ok=True)
cfg.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
log.info(f"ollama base url: {cfg.OLLAMA_BASE_URL}")
log.info(f"ollama model: {cfg.OLLAMA_MODEL}")
log.info(f"embedding model: {cfg.EMBEDDING_MODEL}")
log.info(f"chunk/overlap/k: {cfg.CHUNK_SIZE}/{cfg.CHUNK_OVERLAP}/{cfg.TOP_K}")
log.info(f"Project root: {PROJECT_ROOT}")

2026-05-18 13:30:54,570 - INFO    | rag | ollama base url: http://localhost:11434
2026-05-18 13:30:54,571 - INFO    | rag | ollama model: gemma-4-e4b:latest
2026-05-18 13:30:54,572 - INFO    | rag | embedding model: embeddinggemma:latest
2026-05-18 13:30:54,573 - INFO    | rag | chunk/overlap/k: 800/120/5
2026-05-18 13:30:54,573 - INFO    | rag | Project root: /home/thimu/github_vs/protoRAG/rag-pipeline


In [9]:
SourceFormat = Literal["pdf", "txt", "md", "html", "docx", "xlsx", "pptx", "csv", "json", "xml", "jsonl", "yaml", "yml", "parquet", "avro", "orc", "tsv", "log"]
ElementType = Literal["text", "table", "list", "code", "heading", "metadata", "other"]

class RagChunk(BaseModel):
    """The atomic unit flowing through retrieval. All parsers produce these."""

    # identity
    chunk_id: str = Field(default_factory=lambda: str(uuid4()))
    content_hash: str = ""

    # content
    text: str

    # provenance - these are citation
    source_path: str
    source_format: SourceFormat

    # optional structured metadata (parser-specific, all optional)
    page_number: int | None = None # pdf
    slide_number: int | None = None # pptx
    sheet_name: str | None = None # excel
    section_heading: str | None = None # general document structure
    section_title: str | None = None # pdf bookmarks, html headings, etc.
    row_range: tuple[int, int] | None = None # csv, excel tables

    element_type: ElementType = "text"

    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    extra: dict[str, Any] = Field(default_factory=dict)

    def model_post_init(self, __context: Any) -> None:
        if not self.content_hash:
            self.content_hash = hashlib.sha256(self.text.encode("utf-8")).hexdigest()[:16]

    def to_langchain_metadata(self) -> dict[str, Any]:
        """Flat, JSON-safe, no-None dict for Chroma / LangChain Document.metadata"""
        md: dict[str, Any] = {
            "chunk_id": self.chunk_id,
            "content_hash": self.content_hash,
            "source_path": self.source_path,
            "source_format": self.source_format,
            "element_type": self.element_type,
            "created_at": self.created_at.isoformat(),
        }
        if self.page_number is not None:
            md["page_number"] = self.page_number
        if self.slide_number is not None:
            md["slide_number"] = self.slide_number
        if self.sheet_name is not None:
            md["sheet_name"] = self.sheet_name
        if self.section_heading is not None:
            md["section_heading"] = self.section_heading
        if self.section_title is not None:
            md["section_title"] = self.section_title
        if self.row_range is not None:
            md["row_range"] = f"{self.row_range[0]}-{self.row_range[1]}"
        return md

In [10]:
# smoke test
if __name__ == "__main__":
    _t = RagChunk(
        text="Section 3.2: All employees must complete annual compliance training.",
        source_path="data/raw/test.pdf",
        source_format="pdf",
        page_number=1,
        element_type="text",
    )
    log.info(f"Schema OK | id: {_t.chunk_id} | hash: {_t.content_hash}")
    log.info(f"Metadata Sample: {_t.to_langchain_metadata()}")

2026-05-18 13:30:55,723 - INFO    | rag | Schema OK | id: 8633ada3-efa6-4ef6-bd91-da4462df55ae | hash: 962912e80c914a92
2026-05-18 13:30:55,723 - INFO    | rag | Metadata Sample: {'chunk_id': '8633ada3-efa6-4ef6-bd91-da4462df55ae', 'content_hash': '962912e80c914a92', 'source_path': 'data/raw/test.pdf', 'source_format': 'pdf', 'element_type': 'text', 'created_at': '2026-05-18T05:30:55.723227+00:00', 'page_number': 1}


In [11]:
## docling parser
from docling.document_converter import DocumentConverter
from langchain_text_splitters import RecursiveCharacterTextSplitter

2026-05-18 13:31:55,728 - INFO    | datasets | PyTorch version 2.11.0 available.


In [12]:
class DoclingParser:
    """Phase 0 parser for PDF/PPTX/DOCX/HTML/MD via IBM Docling.

    Pipeline: file → Docling DocumentConverter → markdown → recursive split → RagChunk[].
    Provenance at this stage is file-level (source_path). Page/slide-level
    provenance is added in Phase 1 with HybridChunker.
    """

    SUPPORTED: dict[str, SourceFormat] = {
        ".pdf": "pdf",
        ".pptx": "pptx",
        ".docx": "docx",
        ".html": "html",
        ".htm": "html",
        ".md": "md",
    }

    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 120):
        self.converter = DocumentConverter()
        # Separators ordered from "strong semantic boundary" → "last resort".
        # Markdown headings come first so chunks rarely cross sections.
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", ". ", " ", ""],
        )

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"DoclingParser does not support {path.suffix}")

        source_format = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[Docling] parsing {path.name} ({source_format}) …")

        try:
            result = self.converter.convert(str(path))
        except Exception as e:
            log.error(f"[Docling] convert failed for {path.name}: {e}")
            return []

        markdown = result.document.export_to_markdown()
        if not markdown.strip():
            log.warning(f"[Docling] {path.name}: empty output")
            return []

        rel_source = self._relative_source(path)
        texts = self.splitter.split_text(markdown)

        chunks = [
            RagChunk(
                text=t,
                source_path=rel_source,
                source_format=source_format,
                element_type="text",
            )
            for t in texts if t.strip()
        ]
        log.info(f"[Docling] {path.name}: {len(chunks)} chunks")
        return chunks

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [13]:
parser = DoclingParser(chunk_size=cfg.CHUNK_SIZE, chunk_overlap=cfg.CHUNK_OVERLAP)

# Find the first PDF/PPTX/DOCX in data/raw/
candidates: list[Path] = []
for ext in (".pdf", ".pptx", ".docx"):
    candidates.extend(cfg.DATA_RAW_DIR.rglob(f"*{ext}"))

if not candidates:
    log.warning("No PDF/PPTX/DOCX found under data/raw/ — drop one in and re-run")
else:
    sample = candidates[0]
    sample_chunks = parser.parse(sample)
    print(f"\nFile           : {sample.name}")
    print(f"Total chunks   : {len(sample_chunks)}")
    if sample_chunks:
        c = sample_chunks[0]
        print(f"\n--- First chunk ---")
        print(f"format         : {c.source_format}")
        print(f"length (chars) : {len(c.text)}")
        print(f"chunk_id       : {c.chunk_id[:8]}…")
        print(f"hash           : {c.content_hash}")
        print(f"\ntext preview:\n{c.text[:400]}")
        print(f"\nmetadata sample:\n{c.to_langchain_metadata()}")

2026-05-18 13:31:57,581 - INFO    | rag | [Docling] parsing CV_Himanshu.pdf (pdf) …
2026-05-18 13:31:57,950 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-18 13:31:59,582 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
2026-05-18 13:32:02,913 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
Could not load the custom kernel for multi-scale deformable attention: CUDA_HOME environment variable is not set. Please set it to your CUDA install root.
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiScaleDeformableAttention/MultiScaleDeformableAttention.so: cannot open shared object file: No such file or directory
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiScaleDeformableAttention/MultiScaleDeformableAttention.so: cannot open shared object file: No such


File           : CV_Himanshu.pdf
Total chunks   : 22

--- First chunk ---
format         : pdf
length (chars) : 670
chunk_id       : 0a1451e7…
hash           : e7e9aac80d9cd653

text preview:


<!-- image -->

## Dr. Himanshu Tiwari, PhD

himanshuhimang@gmail.com

 

LinkedIn

 Google Scholar

 ResearchGate



Webpage

ORCID

## PROFESSIONAL SUMMARY

Data Scientist with a strong foundation in Astronomy &amp; Astrophysics. Experienced in machine learning, LLMs, and agentic AI systems, with hands-on expertise across major cloud platforms. Proficient in designing scalable data pipel

metadata sample:
{'chunk_id': '0a1451e7-081c-41f7-b570-83a54f7c3809', 'content_hash': 'e7e9aac80d9cd653', 'source_path': 'data/raw/pdfs/CV_Himanshu.pdf', 'source_format': 'pdf', 'element_type': 'text', 'created_at': '2026-05-18T05:32:15.064478+00:00'}


In [14]:
## structured data parser
import json
import pandas as pd


class StructuredDataParser:
    """Phase 0 parser for CSV/XLSX/JSON/JSONL/TXT.

    Tabular: row → chunk (configurable batch), context prepended.
    JSON: list-of-records → record-per-chunk; else pretty-printed + split.
    JSONL: line → record → chunk.
    TXT: recursive split.
    """

    SUPPORTED: dict[str, SourceFormat] = {
        ".csv": "csv",
        ".xlsx": "xlsx",
        ".xls": "xlsx",
        ".json": "json",
        ".jsonl": "jsonl",
        ".ndjson": "jsonl",
        ".txt": "txt",
    }

    def __init__(
        self,
        rows_per_chunk: int = 1,
        max_rows_warn: int = 10_000,
        chunk_size: int = 800,
        chunk_overlap: int = 120,
    ):
        self.rows_per_chunk = rows_per_chunk
        self.max_rows_warn = max_rows_warn
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"StructuredDataParser does not support {path.suffix}")

        fmt = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[Structured] parsing {path.name} ({fmt}) …")
        try:
            dispatch = {
                "csv": self._parse_csv,
                "xlsx": self._parse_xlsx,
                "json": self._parse_json,
                "jsonl": self._parse_jsonl,
                "txt": self._parse_txt,
            }
            return dispatch[fmt](path)
        except Exception as e:
            log.error(f"[Structured] failed on {path.name}: {e}")
            return []

    # ---------- CSV / XLSX ----------
    def _parse_csv(self, path: Path) -> list[RagChunk]:
        df = pd.read_csv(path)
        if len(df) > self.max_rows_warn:
            log.warning(f"{path.name}: {len(df)} rows — consider increasing rows_per_chunk")
        return self._rows_to_chunks(df, path, "csv", sheet_name=None)

    def _parse_xlsx(self, path: Path) -> list[RagChunk]:
        chunks: list[RagChunk] = []
        xl = pd.ExcelFile(path)
        for sheet in xl.sheet_names:
            df = xl.parse(sheet)
            if df.empty:
                continue
            chunks.extend(self._rows_to_chunks(df, path, "xlsx", sheet_name=sheet))
        return chunks

    def _rows_to_chunks(
        self,
        df: pd.DataFrame,
        path: Path,
        source_format: SourceFormat,
        sheet_name: str | None,
    ) -> list[RagChunk]:
        rel = self._relative_source(path)
        n = len(df)
        if n == 0:
            return []

        scope = f"File: {path.name}"
        if sheet_name:
            scope += f" | Sheet: {sheet_name}"

        chunks: list[RagChunk] = []
        for start in range(0, n, self.rows_per_chunk):
            end = min(start + self.rows_per_chunk, n)
            sub = df.iloc[start:end]
            row_blocks = [
                "\n".join(f"{col}: {self._format_value(val)}" for col, val in row.items())
                for _, row in sub.iterrows()
            ]
            text = f"{scope} | Rows: {start+1}-{end}\n\n" + "\n\n---\n\n".join(row_blocks)
            chunks.append(RagChunk(
                text=text,
                source_path=rel,
                source_format=source_format,
                sheet_name=sheet_name,
                row_range=(start + 1, end),
                element_type="table",
            ))
        return chunks

    # ---------- JSON / JSONL ----------
    def _parse_json(self, path: Path) -> list[RagChunk]:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        rel = self._relative_source(path)

        # List of dicts → record-per-chunk
        if isinstance(data, list) and data and all(isinstance(x, dict) for x in data):
            return [
                self._record_to_chunk(rec, idx, path, rel, "json")
                for idx, rec in enumerate(data)
            ]

        # Dict with single dominant list field → use that list
        if isinstance(data, dict):
            list_keys = [k for k, v in data.items() if isinstance(v, list) and len(v) > 1]
            if len(list_keys) == 1 and all(isinstance(x, dict) for x in data[list_keys[0]]):
                key = list_keys[0]
                return [
                    self._record_to_chunk(rec, idx, path, rel, "json", parent_key=key)
                    for idx, rec in enumerate(data[key])
                ]

        # Fallback: pretty-print whole document, split if large
        text = f"File: {path.name}\n\n{json.dumps(data, indent=2, ensure_ascii=False)}"
        parts = self.splitter.split_text(text) if len(text) > 2000 else [text]
        return [
            RagChunk(text=p, source_path=rel, source_format="json", element_type="text")
            for p in parts if p.strip()
        ]

    def _parse_jsonl(self, path: Path) -> list[RagChunk]:
        rel = self._relative_source(path)
        chunks: list[RagChunk] = []
        with path.open("r", encoding="utf-8") as f:
            for idx, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    log.warning(f"{path.name}: line {idx+1} not valid JSON, skipping")
                    continue
                chunks.append(self._record_to_chunk(rec, idx, path, rel, "jsonl"))
        if len(chunks) > self.max_rows_warn:
            log.warning(f"{path.name}: {len(chunks)} chunks produced")
        return chunks

    def _record_to_chunk(
        self,
        rec: Any,
        idx: int,
        path: Path,
        rel: str,
        fmt: SourceFormat,
        parent_key: str | None = None,
    ) -> RagChunk:
        if isinstance(rec, dict):
            body = "\n".join(f"{k}: {self._format_value(v)}" for k, v in rec.items())
        else:
            body = json.dumps(rec, ensure_ascii=False)
        scope = f"File: {path.name}"
        if parent_key:
            scope += f" | Field: {parent_key}"
        scope += f" | Record: {idx+1}"
        return RagChunk(
            text=f"{scope}\n\n{body}",
            source_path=rel,
            source_format=fmt,
            row_range=(idx + 1, idx + 1),
            element_type="text",
        )

    # ---------- TXT ----------
    def _parse_txt(self, path: Path) -> list[RagChunk]:
        text = path.read_text(encoding="utf-8")
        if not text.strip():
            return []
        rel = self._relative_source(path)
        return [
            RagChunk(text=t, source_path=rel, source_format="txt", element_type="text")
            for t in self.splitter.split_text(text) if t.strip()
        ]

    # ---------- Helpers ----------
    @staticmethod
    def _format_value(val: Any) -> str:
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return ""
        if isinstance(val, (dict, list)):
            return json.dumps(val, ensure_ascii=False)
        return str(val)

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [15]:
struct_parser = StructuredDataParser(
    rows_per_chunk=1,
    chunk_size=cfg.CHUNK_SIZE,
    chunk_overlap=cfg.CHUNK_OVERLAP,
)

candidates: list[Path] = []
for ext in (".csv", ".xlsx", ".json", ".jsonl", ".txt"):
    candidates.extend(cfg.DATA_RAW_DIR.rglob(f"*{ext}"))

if not candidates:
    log.warning("No CSV/XLSX/JSON/JSONL/TXT found under data/raw/ — drop a few in to test")
else:
    for sample in candidates[:5]:  # preview up to 5 files
        result = struct_parser.parse(sample)
        print(f"\n=== {sample.relative_to(PROJECT_ROOT)} ===")
        print(f"  chunks   : {len(result)}")
        if result:
            c = result[0]
            print(f"  format   : {c.source_format} | element: {c.element_type}")
            print(f"  preview  : {c.text[:300]!r}")
            print(f"  metadata : {c.to_langchain_metadata()}")

2026-05-18 13:32:18,862 - INFO    | rag | [Structured] parsing index.json (json) …
2026-05-18 13:32:18,905 - INFO    | rag | [Structured] parsing Nextcloud-backup-codes.txt (txt) …



=== data/raw/json/index.json ===
  chunks   : 1
  format   : json | element: text
  preview  : 'File: index.json\n\n{\n  "schemaVersion": 2,\n  "mediaType": "application/vnd.oci.image.index.v1+json",\n  "manifests": [\n    {\n      "mediaType": "application/vnd.oci.image.manifest.v1+json",\n      "digest": "sha256:96960a1620798b0d68b7cee150bb242fbdf25bb5265d859912ddf7068d7953aa",\n      "size": 1497\n  '
  metadata : {'chunk_id': 'd43629c6-3fc1-4a25-9307-4a9c4c47dd21', 'content_hash': '016d6521eff4ea1e', 'source_path': 'data/raw/json/index.json', 'source_format': 'json', 'element_type': 'text', 'created_at': '2026-05-18T05:32:18.905534+00:00'}

=== data/raw/txt/Nextcloud-backup-codes.txt ===
  chunks   : 1
  format   : txt | element: text
  preview  : '36rYeSWjqsoaKn6i\nfoPJQBW9YL6CmWGs\nfCW7t3C9fActfsot\nYFgKtEyF5YLZ4wEq\nzkwTNCbeWQSfW8MJ\nfGkKED33kxKnHc8m\nWFr6sJZZmX4XqgWH\nrgaRfsZcz6sdeiz3\ndsDn6kW9ffxMseMR\nZqwXCMFQJnf4atgW'
  metadata : {'chunk_id': '65dbfe0c-870b-48f1-86cf-5e971

In [16]:
[█████░░░░░░░░░░░░░░░░░░░░░░░░░░░░] ~15%

PHASE 0 — Naive RAG ◄── WE ARE HERE
   ✓ Config + schema
   ✓ Parsers (Docling track + structured track)
   ◯ Parser dispatcher          ← next (Piece 5)
   ◯ Embedding + Chroma vector store
   ◯ Top-k retrieval
   ◯ Generation with Ollama (gemma-4-e4b)
   ◯ End-to-end answer with citations

PHASE 1 — Hybrid Retrieval
PHASE 2 — Query Intelligence
PHASE 3 — Agentic Orchestration (LangGraph)
PHASE 4 — Evaluation & Observability
PHASE 5 — Production Hardening (FastAPI service)
PHASE 6 — Cloud & Scale (Docker, Azure/AWS/GCP)

SyntaxError: invalid character '█' (U+2588) (2514301984.py, line 1)

In [17]:
from collections import defaultdict
from itertools import groupby
from tqdm.auto import tqdm


class ParserDispatcher:
    """Routes files to the right parser; aggregates RagChunks across a corpus.

    Extension point: append new parsers to `parsers` — first match wins.
    """

    def __init__(self, parsers: list):
        if not parsers:
            raise ValueError("At least one parser required")
        self.parsers = parsers

    def parse_file(self, path: Path) -> list[RagChunk]:
        for parser in self.parsers:
            if parser.supports(path):
                return parser.parse(path)
        return []

    def parse_directory(
        self,
        directory: Path,
        recursive: bool = True,
    ) -> list[RagChunk]:
        if not directory.exists():
            raise FileNotFoundError(f"Directory not found: {directory}")

        files = self._discover_files(directory, recursive)
        if not files:
            log.warning(f"No supported files under {directory}")
            return []

        log.info(f"Found {len(files)} supported file(s) under {directory.name}/")
        all_chunks: list[RagChunk] = []
        files_per_fmt: dict[str, int] = defaultdict(int)
        chunks_per_fmt: dict[str, int] = defaultdict(int)
        failed: list[str] = []
        seen_hashes: set[str] = set()
        dup_count = 0

        for fp in tqdm(files, desc="Parsing", unit="file"):
            try:
                chunks = self.parse_file(fp)
            except Exception as e:
                log.error(f"Parse error on {fp.name}: {e}")
                failed.append(fp.name)
                continue
            if not chunks:
                continue

            fmt = chunks[0].source_format
            files_per_fmt[fmt] += 1
            for chunk in chunks:
                if chunk.content_hash in seen_hashes:
                    dup_count += 1
                    continue
                seen_hashes.add(chunk.content_hash)
                all_chunks.append(chunk)
                chunks_per_fmt[fmt] += 1

        self._print_summary(files_per_fmt, chunks_per_fmt, failed, dup_count, len(all_chunks))
        return all_chunks

    def _discover_files(self, directory: Path, recursive: bool) -> list[Path]:
        pattern = "**/*" if recursive else "*"
        files: list[Path] = []
        for p in directory.glob(pattern):
            if not p.is_file():
                continue
            if any(part.startswith(".") for part in p.parts):
                continue
            if any(parser.supports(p) for parser in self.parsers):
                files.append(p)
        return sorted(files)

    @staticmethod
    def _print_summary(files_per_fmt, chunks_per_fmt, failed, dup_count, total) -> None:
        log.info("=" * 56)
        log.info(f"{'Format':<10} {'Files':>10} {'Chunks':>12}")
        log.info("-" * 56)
        for fmt in sorted(files_per_fmt):
            log.info(f"{fmt:<10} {files_per_fmt[fmt]:>10} {chunks_per_fmt[fmt]:>12}")
        log.info("-" * 56)
        log.info(f"Total unique chunks : {total}")
        log.info(f"Duplicates skipped  : {dup_count}")
        if failed:
            shown = failed[:5]
            extra = f" (+{len(failed)-5} more)" if len(failed) > 5 else ""
            log.info(f"Failed files        : {len(failed)} — {shown}{extra}")
        log.info("=" * 56)


# --- Cache helpers (so we don't re-parse on every notebook re-run) ---

def save_chunks_cache(chunks: list[RagChunk], path: Path) -> None:
    payload = [c.model_dump(mode="json") for c in chunks]
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    log.info(f"Cached {len(chunks)} chunks → {path.relative_to(PROJECT_ROOT)}")


def load_chunks_cache(path: Path) -> list[RagChunk]:
    if not path.exists():
        return []
    data = json.loads(path.read_text(encoding="utf-8"))
    chunks = [RagChunk(**item) for item in data]
    log.info(f"Loaded {len(chunks)} chunks ← {path.relative_to(PROJECT_ROOT)}")
    return chunks

In [18]:
dispatcher = ParserDispatcher(parsers=[
    DoclingParser(chunk_size=cfg.CHUNK_SIZE, chunk_overlap=cfg.CHUNK_OVERLAP),
    StructuredDataParser(
        rows_per_chunk=1,
        chunk_size=cfg.CHUNK_SIZE,
        chunk_overlap=cfg.CHUNK_OVERLAP,
    ),
])

CACHE_PATH = cfg.DATA_PROCESSED_DIR / "phase0_chunks.json"

# Force re-parse by setting REPARSE = True
REPARSE = True

if not REPARSE and CACHE_PATH.exists():
    all_chunks = load_chunks_cache(CACHE_PATH)
else:
    all_chunks = dispatcher.parse_directory(cfg.DATA_RAW_DIR)
    if all_chunks:
        save_chunks_cache(all_chunks, CACHE_PATH)

# Quick peek: one sample chunk per format
if all_chunks:
    print("\nSample chunk per format:")
    by_fmt = sorted(all_chunks, key=lambda c: c.source_format)
    for fmt, group in groupby(by_fmt, key=lambda c: c.source_format):
        sample = next(group)
        print(f"\n--- {fmt} ---")
        print(f"  source : {sample.source_path}")
        print(f"  chars  : {len(sample.text)}")
        print(f"  text   : {sample.text[:220]!r}")
else:
    log.warning("No chunks produced — make sure data/raw/ contains supported files")

2026-05-18 13:32:46,404 - INFO    | rag | Found 5 supported file(s) under raw/


Parsing:   0%|          | 0/5 [00:00<?, ?file/s]

2026-05-18 13:32:46,408 - INFO    | rag | [Docling] parsing Self-Assessment - Himanshu T.docx (docx) …
2026-05-18 13:32:46,446 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-18 13:32:46,447 - INFO    | docling.pipeline.base_pipeline | Processing document Self-Assessment - Himanshu T.docx
2026-05-18 13:32:46,484 - INFO    | docling.document_converter | Finished converting document Self-Assessment - Himanshu T.docx in 0.08 sec.
2026-05-18 13:32:46,498 - INFO    | rag | [Docling] Self-Assessment - Himanshu T.docx: 10 chunks
2026-05-18 13:32:46,500 - INFO    | rag | [Docling] parsing Stage 1_ AI Onboarding - Himanshu.docx (docx) …
2026-05-18 13:32:46,539 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-18 13:32:46,539 - INFO    | docling.pipeline.base_pipeline | Processing document Stage 1_ AI Onboarding - Himanshu.docx
2026-05-18 13:32:46,598 - INFO    | docling.document_converter | Finished converting document Stage 


Sample chunk per format:

--- docx ---
  source : data/raw/docx/Self-Assessment - Himanshu T.docx
  chars  : 453
  text   : '## AI Engineering Onboarding Assessment\n\n- Purpose\n\nThis assessment evaluates the technical capability, system-level thinking, and engineering maturity of incoming AI interns.\n\nIt is designed to determine readiness to co'

--- json ---
  source : data/raw/json/index.json
  chars  : 309
  text   : 'File: index.json\n\n{\n  "schemaVersion": 2,\n  "mediaType": "application/vnd.oci.image.index.v1+json",\n  "manifests": [\n    {\n      "mediaType": "application/vnd.oci.image.manifest.v1+json",\n      "digest": "sha256:96960a16'

--- pdf ---
  source : data/raw/pdfs/CV_Himanshu.pdf
  chars  : 670
  text   : '\uf0e0\n\n<!-- image -->\n\n## Dr. Himanshu Tiwari, PhD\n\nhimanshuhimang@gmail.com\n\n\uf0e0 \uf0e1\n\nLinkedIn\n\n\ue9d4 Google Scholar\n\n\ue95e ResearchGate\n\n\ue9d9\n\nWebpage\n\nORCID\n\n## PROFESSIONAL SUMMARY\n\nData Scientist with a strong foundati

In [19]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings


COLLECTION_NAME = "rag_phase0"

# 1) Embedding function — local, via Ollama
embeddings = OllamaEmbeddings(
    model=cfg.EMBEDDING_MODEL,
    base_url=cfg.OLLAMA_BASE_URL,
)

# Probe: confirm the model is reachable and capture dimension
_probe = embeddings.embed_query("hello world")
EMBED_DIM = len(_probe)
log.info(f"Embedding model '{cfg.EMBEDDING_MODEL}' OK → dim={EMBED_DIM}")


# 2) RagChunk → LangChain Document at the boundary
def chunks_to_documents(chunks: list[RagChunk]) -> tuple[list[Document], list[str]]:
    docs = [
        Document(page_content=c.text, metadata=c.to_langchain_metadata())
        for c in chunks
    ]
    ids = [c.chunk_id for c in chunks]
    return docs, ids


# 3) Open (or create) the persistent Chroma collection
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(cfg.CHROMA_PERSIST_DIR),
)

existing_count = vectorstore._collection.count()
log.info(f"Chroma '{COLLECTION_NAME}': {existing_count} vectors present")


# 4) Idempotent ingest — embed only NEW chunks
REINDEX = False  # flip to True to wipe and rebuild from scratch

if REINDEX and existing_count > 0:
    log.warning(f"REINDEX=True → wiping {existing_count} vectors")
    vectorstore.delete_collection()
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(cfg.CHROMA_PERSIST_DIR),
    )
    existing_count = 0

if not all_chunks:
    log.warning("`all_chunks` is empty — run Piece 5 first, or load from cache")
else:
    docs, ids = chunks_to_documents(all_chunks)

    # Skip chunks already indexed (by chunk_id)
    if existing_count > 0:
        existing_ids = set(vectorstore.get(include=[])["ids"])
        to_add = [(d, i) for d, i in zip(docs, ids) if i not in existing_ids]
    else:
        to_add = list(zip(docs, ids))

    log.info(
        f"To embed: {len(to_add)} new ("
        f"{len(docs) - len(to_add)} already present)"
    )

    # Batch for progress visibility — Ollama on CPU can be slow at first
    BATCH_SIZE = 64
    for i in tqdm(range(0, len(to_add), BATCH_SIZE), desc="Embedding", unit="batch"):
        batch = to_add[i : i + BATCH_SIZE]
        if not batch:
            continue
        b_docs = [p[0] for p in batch]
        b_ids = [p[1] for p in batch]
        vectorstore.add_documents(documents=b_docs, ids=b_ids)

    final_count = vectorstore._collection.count()
    log.info(f"Chroma '{COLLECTION_NAME}': {final_count} vectors total")

2026-05-18 13:33:02,833 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:33:02,833 - INFO    | rag | Embedding model 'embeddinggemma:latest' OK → dim=768
2026-05-18 13:33:03,384 - INFO    | chromadb.telemetry.product.posthog | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-05-18 13:33:03,821 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-05-18 13:33:03,823 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
2026-05-18 13:33:03,829 - INFO    | rag | Chroma 'rag_phase0': 47 vectors present
2026-05-18 13:33:03,830 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argum

Embedding:   0%|          | 0/1 [00:00<?, ?batch/s]

2026-05-18 13:33:06,052 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:33:06,255 - INFO    | rag | Chroma 'rag_phase0': 94 vectors total


In [20]:
def search_and_show(query: str, k: int | None = None) -> list[tuple[Document, float]]:
    k = k or cfg.TOP_K
    results = vectorstore.similarity_search_with_score(query, k=k)

    print(f"\n🔎 Query: {query!r}")
    print(f"Top {k} (Chroma cosine distance — lower = closer):\n")

    for rank, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc_bits = []
        if md.get("page_number"):   loc_bits.append(f"p.{md['page_number']}")
        if md.get("slide_number"):  loc_bits.append(f"slide {md['slide_number']}")
        if md.get("sheet_name"):    loc_bits.append(f"sheet '{md['sheet_name']}'")
        if md.get("row_range"):     loc_bits.append(f"rows {md['row_range']}")
        if md.get("section_title"): loc_bits.append(f"§ {md['section_title']}")
        loc = "  |  ".join(loc_bits)

        print(f"#{rank}  score={score:.4f}  [{md['source_format']}]  {md['source_path']}")
        if loc:
            print(f"     {loc}")
        preview = doc.page_content[:300].replace("\n", " ")
        print(f"     {preview!r}\n")

    return results


# Change this to something that actually appears in YOUR corpus
QUERY = "Himanshu Tiwari profession?"
_ = search_and_show(QUERY)

2026-05-18 13:33:06,506 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:33:06,508 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔎 Query: 'Himanshu Tiwari profession?'
Top 5 (Chroma cosine distance — lower = closer):

#1  score=0.8626  [pdf]  data/raw/pdfs/CV_Himanshu.pdf
     '\uf0e0  <!-- image -->  ## Dr. Himanshu Tiwari, PhD  himanshuhimang@gmail.com  \uf0e0 \uf0e1  LinkedIn  \ue9d4 Google Scholar  \ue95e ResearchGate  \ue9d9  Webpage  ORCID  ## PROFESSIONAL SUMMARY  Data Scientist with a strong foundation in Astronomy &amp; Astrophysics. Experienced in machine learning, LLMs, and agentic AI system'

#2  score=0.8626  [pdf]  data/raw/pdfs/CV_Himanshu.pdf
     '\uf0e0  <!-- image -->  ## Dr. Himanshu Tiwari, PhD  himanshuhimang@gmail.com  \uf0e0 \uf0e1  LinkedIn  \ue9d4 Google Scholar  \ue95e ResearchGate  \ue9d9  Webpage  ORCID  ## PROFESSIONAL SUMMARY  Data Scientist with a strong foundation in Astronomy &amp; Astrophysics. Experienced in machine learning, LLMs, and agentic AI system'

#3  score=1.2425  [pdf]  data/raw/pdfs/CV_Himanshu.pdf
     '## DATA SCIENTIST , NRI Australia &amp; New Zealand  July. 20

In [21]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(
    model=cfg.OLLAMA_MODEL,
    base_url=cfg.OLLAMA_BASE_URL,
    temperature=cfg.LLM_TEMPERATURE,
    num_ctx=cfg.LLM_NUM_CTX,
)

_smoke = llm.invoke("Reply with exactly: pong")
log.info(f"LLM '{cfg.OLLAMA_MODEL}' OK → {_smoke.content!r}")

2026-05-18 13:35:04,159 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:35:04,207 - INFO    | rag | LLM 'gemma-4-e4b:latest' OK → 'pong'


In [22]:
SYSTEM_PROMPT = """You are a precise assistant for company policy and internal documentation questions.

Rules:
1. Use ONLY the information in the CONTEXT below. Do not use outside knowledge.
2. If the answer is not in the context, reply exactly: "I don't have that information in the provided documents."
3. Cite every claim with the source tag in square brackets, e.g. [1] or [2, 3].
4. Quote short passages verbatim when they are decisive (e.g., policy clauses, IDs, dates).
5. Be concise. Do not pad or speculate."""


def _format_location(md: dict) -> str:
    bits = []
    if md.get("page_number"):   bits.append(f"p.{md['page_number']}")
    if md.get("slide_number"):  bits.append(f"slide {md['slide_number']}")
    if md.get("sheet_name"):    bits.append(f"sheet '{md['sheet_name']}'")
    if md.get("row_range"):     bits.append(f"rows {md['row_range']}")
    if md.get("section_title"): bits.append(f"§ {md['section_title']}")
    return " | ".join(bits)


def build_context(results: list[tuple[Document, float]]) -> tuple[str, list[dict]]:
    """Format retrieved chunks as numbered context + return a citation table."""
    blocks: list[str] = []
    citations: list[dict] = []
    for i, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc = _format_location(md)
        header = f"[{i}] {md['source_path']}" + (f" | {loc}" if loc else "")
        blocks.append(f"{header}\n{doc.page_content}")
        citations.append({
            "tag": i,
            "source_path": md["source_path"],
            "source_format": md["source_format"],
            "location": loc,
            "score": float(score),
            "chunk_id": md.get("chunk_id"),
        })
    return "\n\n---\n\n".join(blocks), citations


def answer(query: str, k: int | None = None, show_context: bool = False) -> dict:
    """End-to-end RAG: retrieve → build prompt → generate → return structured result."""
    k = k or cfg.TOP_K
    results = vectorstore.similarity_search_with_score(query, k=k)

    if not results:
        return {
            "question": query,
            "answer": "I don't have any relevant documents indexed.",
            "citations": [],
        }

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])

    out = {
        "question": query,
        "answer": response.content,
        "citations": citations,
    }
    if show_context:
        out["context"] = context_block
    return out


def pretty_print(result: dict) -> None:
    print(f"{result['question']}\n")
    print(f"{result['answer']}\n")
    if result["citations"]:
        print("Sources:")
        for c in result["citations"]:
            loc = f"  |  {c['location']}" if c["location"] else ""
            print(f"   [{c['tag']}] {c['source_path']}{loc}   (dist={c['score']:.3f})")

In [23]:
# Try a query that should be answerable from YOUR corpus
QUERY = "How Himanshu assessed himself in the self-assessment?"

result = answer(QUERY)
pretty_print(result)

# Sanity check the model honours "I don't know"
print("\n" + "=" * 60 + "\n")
nonsense = answer("what's the recipe for chocolate cake")
pretty_print(nonsense)

2026-05-18 13:35:05,766 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:35:06,616 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:35:20,999 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


How Himanshu assessed himself in the self-assessment?

Himanshu's self-rating for the question "How do you balance cost and accuracy in LLM-based systems?" was "2" [5].

Sources:
   [1] data/raw/pdfs/CV_Himanshu.pdf   (dist=1.218)
   [2] data/raw/pdfs/CV_Himanshu.pdf   (dist=1.218)
   [3] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx   (dist=1.262)
   [4] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx   (dist=1.262)
   [5] data/raw/docx/Self-Assessment - Himanshu T.docx   (dist=1.269)




2026-05-18 13:35:21,470 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


what's the recipe for chocolate cake

I don't have that information in the provided documents.

Sources:
   [1] data/raw/pdfs/CV_Himanshu.pdf   (dist=1.437)
   [2] data/raw/pdfs/CV_Himanshu.pdf   (dist=1.437)
   [3] data/raw/pdfs/CV_Himanshu.pdf   (dist=1.510)
   [4] data/raw/pdfs/CV_Himanshu.pdf   (dist=1.510)
   [5] data/raw/pdfs/CV_Himanshu.pdf   (dist=1.536)


[Files in data/raw/]
        ↓  ParserDispatcher
   [RagChunk[]]
        ↓  OllamaEmbeddings (embeddinggemma)
   [Chroma index]
        ↓  similarity_search_with_score
   [Top-k chunks]
        ↓  build_context → SYSTEM_PROMPT
   [Gemma (gemma-4-e4b)]
        ↓
   [Answer + citations]

In [ ]:
[████████████░░░░░░░░░░░░░░░░░░░░░░] ~35%

✓ PHASE 0 — Naive RAG     (DONE — your baseline)
  → PHASE 1 — Hybrid Retrieval ◄── NEXT
    PHASE 2 — Query Intelligence
    PHASE 3 — Agentic Orchestration
    PHASE 4 — Evaluation & Observability
    PHASE 5 — Production Hardening
    PHASE 6 — Cloud & Scale

In [24]:
# phase 1
# hybrid retrieval

In [25]:
import re
from rank_bm25 import BM25Okapi


class BM25Retriever:
    """In-memory BM25 sparse retriever over RagChunks.

    Returns LangChain Documents so it composes with the dense retriever
    under the same interface. Score is BM25 (higher = better) — opposite
    of Chroma distance (lower = better). RRF fusion in the next piece
    sidesteps this by using rank, not raw score.
    """

    # Keep hyphens (vendor IDs like V-001), drop other punctuation.
    _TOKEN_RE = re.compile(r"[^\w\s\-]")

    def __init__(self, chunks: list[RagChunk], min_token_len: int = 2):
        if not chunks:
            raise ValueError("BM25Retriever needs at least one chunk")
        self.chunks = chunks
        self.min_token_len = min_token_len

        log.info(f"[BM25] tokenizing {len(chunks)} chunks …")
        self._tokenized_corpus = [self._tokenize(c.text) for c in chunks]
        self.bm25 = BM25Okapi(self._tokenized_corpus)

        # Pre-build LangChain Documents so retrieval is just a lookup
        self._docs = [
            Document(page_content=c.text, metadata=c.to_langchain_metadata())
            for c in chunks
        ]
        avg_tokens = sum(len(t) for t in self._tokenized_corpus) / len(chunks)
        log.info(f"[BM25] index ready | {len(chunks)} docs | avg {avg_tokens:.0f} tokens/doc")

    def _tokenize(self, text: str) -> list[str]:
        text = text.lower()
        text = self._TOKEN_RE.sub(" ", text)
        return [t for t in text.split() if len(t) >= self.min_token_len]

    def retrieve(
        self,
        query: str,
        k: int = 10,
        min_score: float = 0.0,
    ) -> list[tuple[Document, float]]:
        tokens = self._tokenize(query)
        if not tokens:
            return []
        scores = self.bm25.get_scores(tokens)
        # Argsort top-k
        ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return [
            (self._docs[i], float(scores[i]))
            for i in ranked_idx
            if scores[i] > min_score
        ]



In [26]:

# Build the index from chunks already in memory (or load from cache first if needed)
if not all_chunks:
    all_chunks = load_chunks_cache(cfg.DATA_PROCESSED_DIR / "phase0_chunks.json")

bm25_retriever = BM25Retriever(all_chunks)

2026-05-18 13:35:31,164 - INFO    | rag | [BM25] tokenizing 47 chunks …
2026-05-18 13:35:31,167 - INFO    | rag | [BM25] index ready | 47 docs | avg 69 tokens/doc


In [27]:
def show_results(title: str, results: list[tuple[Document, float]], score_label: str) -> None:
    print(f"\n--- {title} ---")
    if not results:
        print("  (no results)")
        return
    for rank, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc = _format_location(md)
        loc_str = f"  |  {loc}" if loc else ""
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"  #{rank}  {score_label}={score:.3f}  {md['source_path']}{loc_str}")
        print(f"        {preview!r}")


# Try BOTH a literal/keyword query and a semantic/paraphrase query
TEST_QUERIES = [
    "JADES-FRESCO",                              # literal ID — BM25 should win
    "Self-Assessment",        # exact-ish phrase — both should hit
    "How himanshu assessed himself?",   # paraphrase — dense should win
]

for q in TEST_QUERIES:
    print("=" * 70)
    print(f"🔎 Query: {q!r}")

    dense_results = vectorstore.similarity_search_with_score(q, k=5)
    bm25_results = bm25_retriever.retrieve(q, k=5)

    show_results("DENSE (Chroma, embeddinggemma)", dense_results, "dist")
    show_results("BM25  (rank_bm25)",              bm25_results,  "bm25")

2026-05-18 13:35:35,228 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


🔎 Query: 'JADES-FRESCO'

--- DENSE (Chroma, embeddinggemma) ---
  #1  dist=1.223  data/raw/pdfs/CV_Himanshu.pdf
        "## PRESENTATIONS N' OUTREACH"
  #2  dist=1.223  data/raw/pdfs/CV_Himanshu.pdf
        "## PRESENTATIONS N' OUTREACH"
  #3  dist=1.304  data/raw/txt/Nextcloud-backup-codes.txt
        '36rYeSWjqsoaKn6i foPJQBW9YL6CmWGs fCW7t3C9fActfsot YFgKtEyF5YLZ4wEq zkwTNCbeWQSfW8MJ fGkKED33kxKnHc8m WFr6sJZZmX4XqgWH r'
  #4  dist=1.304  data/raw/txt/Nextcloud-backup-codes.txt
        '36rYeSWjqsoaKn6i foPJQBW9YL6CmWGs fCW7t3C9fActfsot YFgKtEyF5YLZ4wEq zkwTNCbeWQSfW8MJ fGkKED33kxKnHc8m WFr6sJZZmX4XqgWH r'
  #5  dist=1.338  data/raw/pdfs/CV_Himanshu.pdf
        '## FAST RADIO BURSTS (FRBS)  <!-- image -->  1. Modelling the energy distribution in CHIME/FRB Catalog-1 Siddhartha Bhat'

--- BM25  (rank_bm25) ---
  #1  bm25=3.419  data/raw/pdfs/CV_Himanshu.pdf
        '## RESEARCH ASSISTANT , Curtin Institute of Radio Astronomy  Oct. 2024 - April 2025  - · Study of EoR analogous extreme-'

2026-05-18 13:35:38,809 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"



--- DENSE (Chroma, embeddinggemma) ---
  #1  dist=1.181  data/raw/docx/Self-Assessment - Himanshu T.docx
        'Expected Competency:  - Task decomposition - Sequential reasoning - Workflow design  Self-Rating (1–5): \\_\\_\\_2\\_\\_\\_\\_ '
  #2  dist=1.181  data/raw/docx/Self-Assessment - Himanshu T.docx
        'Expected Competency:  - Task decomposition - Sequential reasoning - Workflow design  Self-Rating (1–5): \\_\\_\\_2\\_\\_\\_\\_ '
  #3  dist=1.205  data/raw/docx/Self-Assessment - Himanshu T.docx
        'Self-Rating (1–5): \\_\\_2\\_\\_\\_\\_\\_  Discussion:  - Where does asynchronous processing improve performance in AI systems?'
  #4  dist=1.205  data/raw/docx/Self-Assessment - Himanshu T.docx
        'Self-Rating (1–5): \\_\\_2\\_\\_\\_\\_\\_  Discussion:  - Where does asynchronous processing improve performance in AI systems?'
  #5  dist=1.223  data/raw/docx/Self-Assessment - Himanshu T.docx
        '## AI Engineering Onboarding Assessment  - Purpose  This assessment e

2026-05-18 13:35:41,549 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"



--- DENSE (Chroma, embeddinggemma) ---
  #1  dist=1.228  data/raw/docx/Self-Assessment - Himanshu T.docx
        '- How do you balance cost and accuracy in LLM-based systems?  Self-Rating (1–5): \\_\\_2\\_\\_\\_\\_\\_  Discussion:  - How wou'
  #2  dist=1.228  data/raw/docx/Self-Assessment - Himanshu T.docx
        '- How do you balance cost and accuracy in LLM-based systems?  Self-Rating (1–5): \\_\\_2\\_\\_\\_\\_\\_  Discussion:  - How wou'
  #3  dist=1.294  data/raw/docx/Self-Assessment - Himanshu T.docx
        'Discussion:  - How would you evaluate a production RAG system?  Self-Rating (1–5): \\_\\_\\_1\\_\\_\\_\\_  Discussion:  - How d'
  #4  dist=1.294  data/raw/docx/Self-Assessment - Himanshu T.docx
        'Discussion:  - How would you evaluate a production RAG system?  Self-Rating (1–5): \\_\\_\\_1\\_\\_\\_\\_  Discussion:  - How d'
  #5  dist=1.313  data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx
        '## SignChain AI Evolution - RAG &amp; Agentic  Assigned to: Hima

In [28]:
## combining both retrievers: 
## RRF_score (doc) = Σ over retrievers:  weight / (k + rank_of_doc_in_that_retriever)

In [29]:
def reciprocal_rank_fusion(
    rankings: list[list[Document]],
    k: int = 60,
    weights: list[float] | None = None,
) -> list[tuple[Document, float]]:
    """Fuse multiple ranked Document lists via Reciprocal Rank Fusion.

    Uses rank position only — immune to score-scale mismatches between
    retrievers. Documents are deduped by chunk_id from their metadata.
    """
    if weights is None:
        weights = [1.0] * len(rankings)
    if len(weights) != len(rankings):
        raise ValueError("weights must match number of rankings")

    scores: dict[str, float] = defaultdict(float)
    doc_lookup: dict[str, Document] = {}

    for ranking, w in zip(rankings, weights):
        for rank, doc in enumerate(ranking, start=1):
            doc_id = doc.metadata.get("chunk_id") or doc.metadata.get("content_hash")
            if not doc_id:
                continue
            scores[doc_id] += w / (k + rank)
            doc_lookup.setdefault(doc_id, doc)

    return sorted(
        ((doc_lookup[did], s) for did, s in scores.items()),
        key=lambda x: x[1],
        reverse=True,
    )


class EnsembleRetriever:
    """Hybrid retriever = dense ⊕ BM25, fused via RRF.

    fetch_k: how many to pull from EACH base retriever before fusion, default 20.
    top_k:   how many to return after fusion, default 5.
    rrf_k:   the "k" parameter for RRF fusion, default 60 (per the research paper's recommendation).

    Weights are per-retriever. Equal weights (1.0, 1.0) is a strong default.
    Bump dense weight up if your corpus is paraphrase-heavy; bump BM25 up
    if it's full of IDs / codes / exact references.
    """

    def __init__(
        self,
        dense_store: Chroma,
        sparse_retriever: BM25Retriever,
        fetch_k: int = 20, 
        weights: tuple[float, float] = (1.0, 1.0),
        rrf_k: int = 60,
    ):
        self.dense_store = dense_store
        self.sparse_retriever = sparse_retriever
        self.fetch_k = fetch_k
        self.weights = list(weights)
        self.rrf_k = rrf_k

    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Document, float]]:
        # Dense: we only need order, not the distance, for RRF
        dense_docs = self.dense_store.similarity_search(query, k=self.fetch_k)

        # Sparse: returns (doc, bm25_score); keep just docs, in rank order
        sparse_pairs = self.sparse_retriever.retrieve(query, k=self.fetch_k)
        sparse_docs = [d for d, _ in sparse_pairs]

        fused = reciprocal_rank_fusion(
            rankings=[dense_docs, sparse_docs],
            k=self.rrf_k,
            weights=self.weights,
        )
        return fused[:top_k]




In [30]:
ensemble = EnsembleRetriever(
    dense_store=vectorstore,
    sparse_retriever=bm25_retriever,
    fetch_k=20,
    weights=(1.0, 1.0),
)
log.info(f"[Ensemble] ready | fetch_k={ensemble.fetch_k} | weights={ensemble.weights}")

2026-05-18 13:35:43,728 - INFO    | rag | [Ensemble] ready | fetch_k=20 | weights=[1.0, 1.0]


In [31]:
## comparing dense vs. bm25 vs hybrid

In [32]:
# Redefine pretty_print to accept a score-label (replaces the Phase 0 version)
def pretty_print(result: dict, score_label: str = "score") -> None:
    print(f"{result['question']}\n")
    print(f"{result['answer']}\n")
    if result["citations"]:
        print("Sources:")
        for c in result["citations"]:
            loc = f"  |  {c['location']}" if c["location"] else ""
            print(f"   [{c['tag']}] {c['source_path']}{loc}   ({score_label}={c['score']:.3f})")


def answer_phase1(
    query: str,
    k: int | None = None,
    retriever: str = "hybrid",  # "dense" | "bm25" | "hybrid"
) -> dict:
    k = k or cfg.TOP_K

    if retriever == "dense":
        results = vectorstore.similarity_search_with_score(query, k=k)
    elif retriever == "bm25":
        results = bm25_retriever.retrieve(query, k=k)
    elif retriever == "hybrid":
        results = ensemble.retrieve(query, top_k=k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if not results:
        return {"question": query, "answer": "No relevant docs.", "citations": [], "retriever": retriever}

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])
    return {
        "question": query,
        "answer": response.content,
        "citations": citations,
        "retriever": retriever,
    }


SCORE_LABELS = {"dense": "dist", "bm25": "bm25", "hybrid": "rrf"}

# Pick ONE query from your corpus and watch how the three retrievers differ
TEST_QUERY = "AI engineering understanding"

for r in ("dense", "bm25", "hybrid"):
    print("=" * 72)
    print(f"RETRIEVER: {r.upper()}")
    print("=" * 72)
    pretty_print(answer_phase1(TEST_QUERY, retriever=r), score_label=SCORE_LABELS[r])
    print()

2026-05-18 13:35:48,403 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


RETRIEVER: DENSE


2026-05-18 13:35:49,164 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


AI engineering understanding

AI engineering understanding, as assessed for incoming AI interns, evaluates technical capability, system-level thinking, and engineering maturity [1, 2].

The assessment determines readiness to contribute to several areas, including:
*   Retrieval and knowledge-grounded AI systems [1, 2].
*   Agentic AI workflows and orchestration [1, 2].
*   Document intelligence and large-scale data pipelines [1, 2].
*   Production-grade, API-driven AI architectures [1, 2].

Key technical competencies assessed include:

*   **Data Processing:** Processing large, unstructured data (documents, logs, drawings) efficiently and reliably, requiring skills in "Incremental processing (streaming, batching)," "Data transformation and structuring," and "Memory-safe processing strategies" [3, 4].
*   **External Services:** Managing dependencies on external services (LLM APIs and embedding services), focusing on "Async vs synchronous execution," "Concurrency and throughput optimizat

2026-05-18 13:36:28,351 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:37:06,966 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


AI engineering understanding

AI engineering understanding covers several areas, including system architecture, model mechanics, and practical implementation skills [1, 2, 5].

**System Architecture and Design:**
*   AI pipelines must be modular and testable, consisting of multiple stages (ingestion, processing, retrieval, generation) [2].
*   Expected competencies include "Pipeline modularization," "Separation of concerns," and "Maintainability and testability" [2].
*   Experience includes designing scalable data pipelines and building end-to-end data products [4].
*   Practical experience involves developing "Agentic AI-solution[s]" that leverage LLMs, RAG, and fine-tuning [3].

**Model and AI Mechanics:**
*   Modern AI systems are built on transformer architectures [2, 5].
*   Understanding model behavior is critical, even when using pre-trained systems [2].
*   Key technical knowledge areas include:
    *   High-level understanding of attention, tokenization, and context limits [5]

2026-05-18 13:37:07,695 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


AI engineering understanding

AI engineering understanding, based on the assessment, requires knowledge across system architecture, data processing, model mechanics, and operational reliability [1].

**1. System Architecture and Pipelines**
*   **Design:** AI pipelines must be "modular and testable" [2].
*   **Competencies:** Expected skills include "Pipeline modularization," "Separation of concerns," and "Maintainability and testability" [2].
*   **Workflows:** Systems increasingly rely on "multi-step workflows rather than single prompts" [3]. Expected competencies for these workflows are "Task decomposition," "Sequential reasoning," and "Workflow design" [3].

**2. Data Processing and Retrieval**
*   **Data Handling:** AI systems must process "large, unstructured data (documents, logs, drawings) efficiently and reliably" [4].
*   **Competencies:** This requires knowledge of "Incremental processing (streaming, batching)," "Data transformation and structuring," and "Memory-safe process

In [33]:
## reranker: pulling from top 20 retrieved chuncks

In [34]:
from FlagEmbedding import FlagReranker


class Reranker:
    """Thin wrapper around BGE cross-encoder rerankers."""

    def __init__(
        self,
        model_name: str = "BAAI/bge-reranker-base",
        use_fp16: bool = True,
        normalize: bool = True,
    ):
        log.info(f"[Reranker] loading {model_name} (first call downloads ~1GB)…")
        self.model = FlagReranker(model_name, use_fp16=use_fp16)
        self.model_name = model_name
        self.normalize = normalize  # sigmoid → 0..1 scores, intuitive thresholds
        log.info(f"[Reranker] ready")

    def score(self, query: str, docs: list[Document]) -> list[float]:
        if not docs:
            return []
        pairs = [[query, d.page_content] for d in docs]
        out = self.model.compute_score(pairs, normalize=self.normalize)
        # FlagReranker returns float for a single pair, list for multiple
        if isinstance(out, float):
            return [out]
        return [float(x) for x in out]


class RerankedRetriever:
    """Wrap any base retriever; over-fetch then rerank with a cross-encoder.

    Pipeline:  base.retrieve(top_k=fetch_k)  →  rerank  →  top_k

    Cuts the noise that hybrid retrieval inevitably brings in.
    """

    def __init__(
        self,
        base_retriever,
        reranker: Reranker,
        fetch_k: int = 20,
        min_score: float | None = None,  # set in Phase 4 eval, not by vibes
    ):
        self.base = base_retriever
        self.reranker = reranker
        self.fetch_k = fetch_k
        self.min_score = min_score

    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Document, float]]:
        candidates = self.base.retrieve(query, top_k=self.fetch_k)
        if not candidates:
            return []

        docs = [d for d, _ in candidates]
        scores = self.reranker.score(query, docs)

        scored = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
        if self.min_score is not None:
            scored = [(d, s) for d, s in scored if s >= self.min_score]
        return scored[:top_k]


In [35]:
# download the reranker model
reranker = Reranker(model_name="BAAI/bge-reranker-base", use_fp16=True, normalize=True)

hybrid_reranked = RerankedRetriever(
    base_retriever=ensemble,    # our hybrid retriever from Cell 19
    reranker=reranker,
    fetch_k=20,
    min_score=None,             # learn this from eval, not intuition
)
log.info(f"[Pipeline] hybrid+rerank ready | fetch_k={hybrid_reranked.fetch_k}")

2026-05-18 13:38:09,940 - INFO    | rag | [Reranker] loading BAAI/bge-reranker-base (first call downloads ~1GB)…
2026-05-18 13:38:15,503 - INFO    | rag | [Reranker] ready
2026-05-18 13:38:15,504 - INFO    | rag | [Pipeline] hybrid+rerank ready | fetch_k=20


In [36]:
## testing hybrid vs hybrid+rerank on the same query

In [37]:
def answer_phase1(
    query: str,
    k: int | None = None,
    retriever: str = "hybrid_reranked",   # new default
) -> dict:
    k = k or cfg.TOP_K

    if retriever == "dense":
        results = vectorstore.similarity_search_with_score(query, k=k)
    elif retriever == "bm25":
        results = bm25_retriever.retrieve(query, k=k)
    elif retriever == "hybrid":
        results = ensemble.retrieve(query, top_k=k)
    elif retriever == "hybrid_reranked":
        results = hybrid_reranked.retrieve(query, top_k=k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if not results:
        return {"question": query, "answer": "No relevant docs.", "citations": [], "retriever": retriever}

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])
    return {
        "question": query,
        "answer": response.content,
        "citations": citations,
        "retriever": retriever,
    }


SCORE_LABELS = {
    "dense": "dist", "bm25": "bm25", "hybrid": "rrf", "hybrid_reranked": "rerank",
}

# Use a query where the right answer needs precision, not just recall
TEST_QUERY = "where Organised Intra-House social, sporting events?"

for r in ("hybrid", "hybrid_reranked"):
    print("=" * 72)
    print(f"RETRIEVER: {r.upper()}")
    print("=" * 72)
    pretty_print(answer_phase1(TEST_QUERY, retriever=r), score_label=SCORE_LABELS[r])
    print()

RETRIEVER: HYBRID


2026-05-18 13:38:16,087 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:38:16,962 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:38:30,513 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


where Organised Intra-House social, sporting events?

The events were organized "while my stay at Unilodge" [1, 4].

Sources:
   [1] data/raw/pdfs/CV_Himanshu.pdf   (rrf=0.033)
   [2] data/raw/pdfs/CV_Himanshu.pdf   (rrf=0.031)
   [3] data/raw/pdfs/CV_Himanshu.pdf   (rrf=0.030)
   [4] data/raw/pdfs/CV_Himanshu.pdf   (rrf=0.016)
   [5] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx   (rrf=0.016)

RETRIEVER: HYBRID_RERANKED


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2026-05-18 13:38:44,408 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


where Organised Intra-House social, sporting events?

The events were organized "while my stay at Unilodge" [1, 2].

Sources:
   [1] data/raw/pdfs/CV_Himanshu.pdf   (rerank=0.995)
   [2] data/raw/pdfs/CV_Himanshu.pdf   (rerank=0.995)
   [3] data/raw/pdfs/CV_Himanshu.pdf   (rerank=0.003)
   [4] data/raw/pdfs/CV_Himanshu.pdf   (rerank=0.003)
   [5] data/raw/pdfs/CV_Himanshu.pdf   (rerank=0.000)



In [38]:
pip install "docling-core[chunking]" --break-system-packages

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


## phase 1: hybrid chunker

In [39]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer


class DoclingHybridParser:
    """Phase 1: structure-aware chunking with real provenance.

    vs DoclingParser (Phase 0):
      • chunks respect headings/tables/sections (no mid-table slicing)
      • real page_number (PDF) / slide_number (PPTX)
      • section hierarchy captured in section_title
      • heading context prepended to text → better embeddings
    """

    SUPPORTED: dict[str, SourceFormat] = {
        ".pdf": "pdf", ".pptx": "pptx", ".docx": "docx",
        ".html": "html", ".htm": "html", ".md": "md",
    }

    def __init__(
        self,
        max_tokens: int = 512,
        tokenizer_id: str = "sentence-transformers/all-MiniLM-L6-v2",
        merge_peers: bool = True,
    ):
        self.converter = DocumentConverter()
        tok = HuggingFaceTokenizer(
            tokenizer=AutoTokenizer.from_pretrained(tokenizer_id),
            max_tokens=max_tokens,
        )
        self.chunker = HybridChunker(tokenizer=tok, merge_peers=merge_peers)

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"DoclingHybridParser does not support {path.suffix}")

        fmt = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[DoclingHybrid] parsing {path.name} ({fmt}) …")
        try:
            result = self.converter.convert(str(path))
        except Exception as e:
            log.error(f"[DoclingHybrid] convert failed for {path.name}: {e}")
            return []

        doc = result.document
        rel = self._relative_source(path)
        chunks: list[RagChunk] = []

        for dl_chunk in self.chunker.chunk(dl_doc=doc):
            # contextualize() prepends the heading hierarchy → richer embeddings
            text = self.chunker.contextualize(chunk=dl_chunk)
            if not text or not text.strip():
                continue

            page_no = self._first_page_no(dl_chunk)
            chunks.append(RagChunk(
                text=text,
                source_path=rel,
                source_format=fmt,
                page_number=page_no if fmt == "pdf" else None,
                slide_number=page_no if fmt == "pptx" else None,
                section_title=self._section_title(dl_chunk),
                element_type=self._element_type(dl_chunk),
            ))

        log.info(f"[DoclingHybrid] {path.name}: {len(chunks)} structure-aware chunks")
        return chunks

    @staticmethod
    def _first_page_no(dl_chunk) -> int | None:
        try:
            for item in dl_chunk.meta.doc_items:
                for prov in getattr(item, "prov", []) or []:
                    pn = getattr(prov, "page_no", None)
                    if pn is not None:
                        return int(pn)
        except Exception:
            pass
        return None

    @staticmethod
    def _section_title(dl_chunk) -> str | None:
        try:
            headings = getattr(dl_chunk.meta, "headings", None)
            if headings:
                return " > ".join(h for h in headings if h)[:300]
        except Exception:
            pass
        return None

    @staticmethod
    def _element_type(dl_chunk) -> ElementType:
        try:
            for item in dl_chunk.meta.doc_items:
                label = str(getattr(item, "label", "")).lower()
                if "table" in label: return "table"
                if "list" in label:  return "list"
                if "code" in label:  return "code"
        except Exception:
            pass
        return "text"

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [40]:
## re-parse + cheap peek
dispatcher_v2 = ParserDispatcher(parsers=[
    DoclingHybridParser(max_tokens=512, merge_peers=True),
    StructuredDataParser(
        rows_per_chunk=1,
        chunk_size=cfg.CHUNK_SIZE,
        chunk_overlap=cfg.CHUNK_OVERLAP,
    ),
])

all_chunks_v2 = dispatcher_v2.parse_directory(cfg.DATA_RAW_DIR)
save_chunks_cache(all_chunks_v2, cfg.DATA_PROCESSED_DIR / "phase1_chunks.json")

# Confirm page_number + section_title are now populated on a PDF chunk
pdf_chunks = [c for c in all_chunks_v2 if c.source_format == "pdf"]
if pdf_chunks:
    sample = next((c for c in pdf_chunks if c.page_number is not None), pdf_chunks[0])
    print("Structure-aware PDF chunk:")
    print(f"  page_number   : {sample.page_number}")
    print(f"  section_title : {sample.section_title}")
    print(f"  element_type  : {sample.element_type}")
    print(f"  text preview  : {sample.text[:320]!r}")
else:
    log.warning("No PDF chunks — add a PDF to data/raw/pdfs/ to see page numbers")

/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2026-05-18 13:40:46,300 - INFO    | rag | Found 5 supported file(s) under raw/


Parsing:   0%|          | 0/5 [00:00<?, ?file/s]

2026-05-18 13:40:46,304 - INFO    | rag | [DoclingHybrid] parsing Self-Assessment - Himanshu T.docx (docx) …
2026-05-18 13:40:46,311 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-18 13:40:46,311 - INFO    | docling.pipeline.base_pipeline | Processing document Self-Assessment - Himanshu T.docx
2026-05-18 13:40:46,348 - INFO    | docling.document_converter | Finished converting document Self-Assessment - Himanshu T.docx in 0.04 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (526 > 512). Running this sequence through the model will result in indexing errors
2026-05-18 13:40:46,443 - INFO    | rag | [DoclingHybrid] Self-Assessment - Himanshu T.docx: 3 structure-aware chunks
2026-05-18 13:40:46,444 - INFO    | rag | [DoclingHybrid] parsing Stage 1_ AI Onboarding - Himanshu.docx (docx) …
2026-05-18 13:40:46,448 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-18 13

Structure-aware PDF chunk:
  page_number   : 1
  section_title : None
  element_type  : text
  text preview  : '\uf0e0'


In [41]:
all_chunks = all_chunks_v2  # promote to the working corpus

# 1) Chroma: chunk boundaries changed → full re-embed required
log.warning("Rebuilding Chroma with structure-aware chunks (one-time re-embed)…")
vectorstore.delete_collection()
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(cfg.CHROMA_PERSIST_DIR),
)
docs, ids = chunks_to_documents(all_chunks)
BATCH = 64
for i in tqdm(range(0, len(docs), BATCH), desc="Re-embedding", unit="batch"):
    vectorstore.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])
log.info(f"Chroma rebuilt → {vectorstore._collection.count()} vectors")

# 2) BM25: rebuild from new chunks
bm25_retriever = BM25Retriever(all_chunks)

# 3) Rewire ensemble + reranker (they held refs to the OLD objects)
ensemble = EnsembleRetriever(vectorstore, bm25_retriever, fetch_k=20, weights=(1.0, 1.0))
hybrid_reranked = RerankedRetriever(ensemble, reranker, fetch_k=20)
log.info("Phase 1 pipeline rebuilt — structure-aware end to end")

# 4) Verify: citations should now show p.N and § Section
result = answer_phase1("what is the policy for vendor approval",
                        retriever="hybrid_reranked")
pretty_print(result, score_label="rerank")

2026-05-18 13:41:04,907 - WARNING | rag | Rebuilding Chroma with structure-aware chunks (one-time re-embed)…
2026-05-18 13:41:05,230 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-05-18 13:41:06,018 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Re-embedding:   0%|          | 0/2 [00:00<?, ?batch/s]

2026-05-18 13:41:08,019 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:41:13,819 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:41:15,846 - INFO    | rag | Chroma rebuilt → 79 vectors
2026-05-18 13:41:15,846 - INFO    | rag | [BM25] tokenizing 79 chunks …
2026-05-18 13:41:15,848 - INFO    | rag | [BM25] index ready | 79 docs | avg 43 tokens/doc
2026-05-18 13:41:15,849 - INFO    | rag | Phase 1 pipeline rebuilt — structure-aware end to end
2026-05-18 13:41:15,920 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 13:41:15,921 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
2026-05-18 13:41:17,081 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


what is the policy for vendor approval

I don't have that information in the provided documents.

Sources:
   [1] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx  |  § SignChain AI Evolution - RAG & Agentic > 1. Context & Purpose   (rerank=0.007)
   [2] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx  |  § SignChain AI Evolution - RAG & Agentic   (rerank=0.001)
   [3] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx  |  § SignChain AI Evolution - RAG & Agentic > 7. What Success Looks Like   (rerank=0.000)
   [4] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx  |  § SignChain AI Evolution - RAG & Agentic > 6. Suggest Timeline   (rerank=0.000)
   [5] data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx  |  § SignChain AI Evolution - RAG & Agentic > 10. Important Note   (rerank=0.000)


In [ ]:
[████████████████████░░░░░░░░░░░░░░] ~60%

✓ PHASE 0 — Naive RAG            (baseline)
✓ PHASE 1 — Hybrid Retrieval     (DONE)
    ✓ BM25 sparse retriever
    ✓ Ensemble (dense ⊕ BM25, RRF fusion)
    ✓ Cross-encoder reranker (BGE)
    ✓ Structure-aware chunking + real page/section citations
  → next: PHASE 2 or PHASE 4

## phase 1 and phase 2 done
## moving to phase 4 evaluation & metrics

## adding query intelligence in the next step, once the evaluation framework is in place to show its impact

In [43]:
## eval schema + synthetic dataset creation
import random


class EvalExample(BaseModel):
    """One labelled eval case. Source-path level → survives re-chunking.

    Convention: gold_source_paths == []  means  "system SHOULD refuse / find nothing"
    (used later to test the 'I don't know' path and the min_score threshold).
    """
    question: str
    gold_source_paths: list[str]
    gold_snippets: list[str] = Field(default_factory=list)
    reference_answer: str | None = None
    difficulty: Literal["easy", "medium", "hard"] = "medium"
    notes: str = ""


QGEN_PROMPT = """You are creating evaluation questions for a company's internal-document search system.

Given the SOURCE PASSAGE below, write ONE realistic question an employee would ask, where THIS passage contains the answer.

Rules:
- Answerable using ONLY facts in this passage.
- Do NOT reference "this document/passage/table/above". Ask as if you don't know where the answer lives.
- Prefer specific factual questions (names, numbers, policies, IDs, dates) over vague ones.
- Also copy the SHORT exact answer snippet verbatim from the passage.

Return STRICT JSON, no markdown fences:
{{"question": "...", "answer_snippet": "...", "difficulty": "easy|medium|hard"}}

SOURCE PASSAGE:
{passage}"""


def generate_eval_set(
    chunks: list[RagChunk],
    n: int = 20,
    seed: int = 42,
    min_chunk_chars: int = 200,
) -> list[EvalExample]:
    """LLM-synthesised eval set. CURATE the output by hand afterwards — not optional."""
    rng = random.Random(seed)
    pool = [c for c in chunks if len(c.text) >= min_chunk_chars] or list(chunks)
    sample = rng.sample(pool, min(n, len(pool)))

    examples: list[EvalExample] = []
    for i, ch in enumerate(tqdm(sample, desc="Generating Q", unit="q"), 1):
        prompt = QGEN_PROMPT.format(passage=ch.text[:2000])
        try:
            resp = llm.invoke([HumanMessage(content=prompt)])
            raw = re.sub(r"^```(?:json)?|```$", "", resp.content.strip(),
                         flags=re.MULTILINE).strip()
            data = json.loads(raw)
        except Exception as e:
            log.warning(f"Q{i}: generation/parse failed ({e}); skipping")
            continue

        q = (data.get("question") or "").strip()
        snippet = (data.get("answer_snippet") or "").strip()
        diff = data.get("difficulty", "medium")
        diff = diff if diff in ("easy", "medium", "hard") else "medium"

        if len(q) < 12 or any(bad in q.lower() for bad in (
            "this document", "this passage", "the table", "above", "the text"
        )):
            log.warning(f"Q{i}: degenerate question rejected: {q!r}")
            continue

        examples.append(EvalExample(
            question=q,
            gold_source_paths=[ch.source_path],
            gold_snippets=[snippet] if snippet else [],
            difficulty=diff,
            notes=f"auto-gen from {ch.source_format} chunk {ch.chunk_id[:8]}",
        ))

    log.info(f"Generated {len(examples)} eval examples (requested {n})")
    return examples


def save_eval_set(examples: list[EvalExample], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps([e.model_dump() for e in examples], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    log.info(f"Saved {len(examples)} examples → {path.relative_to(PROJECT_ROOT)}")


def load_eval_set(path: Path) -> list[EvalExample]:
    if not path.exists():
        return []
    return [EvalExample(**d) for d in json.loads(path.read_text(encoding="utf-8"))]

In [44]:
EVAL_DIR = PROJECT_ROOT / "eval"
EVAL_SET_PATH = EVAL_DIR / "eval_set.json"

REGENERATE_EVAL = True  # flip to False AFTER you've curated the set

if REGENERATE_EVAL or not EVAL_SET_PATH.exists():
    eval_set = generate_eval_set(all_chunks, n=20, seed=42)
    save_eval_set(eval_set, EVAL_SET_PATH)
else:
    eval_set = load_eval_set(EVAL_SET_PATH)
    log.info(f"Loaded {len(eval_set)} curated examples")

print(f"\n{len(eval_set)} eval examples — REVIEW THESE:\n")
for i, ex in enumerate(eval_set, 1):
    print(f"[{i}] ({ex.difficulty}) {ex.question}")
    print(f"     gold : {ex.gold_source_paths}")
    print(f"     snip : {ex.gold_snippets}\n")

Generating Q:   0%|          | 0/20 [00:00<?, ?q/s]

2026-05-18 13:49:02,638 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:49:05,646 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:49:08,437 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:49:32,244 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:49:35,126 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:49:58,421 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:50:00,483 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:50:22,146 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-18 13:50:49,451 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "


20 eval examples — REVIEW THESE:

[1] (easy) What are the three capabilities of a Context Retrieval System that should be built?
     gold : ['data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx']
     snip : ['- Retrieve relevant clauses (not just text chunks)\n- Use metadata for better accuracy\n- Improve answer grounding']

[2] (easy) What are the expected competencies for designing a modular pipeline for a knowledge-based AI system?
     gold : ['data/raw/docx/Self-Assessment - Himanshu T.docx']
     snip : ['Pipeline modularization\nSeparation of concerns\nMaintainability and testability']

[3] (easy) What specific types of use cases need working prototypes?
     gold : ['data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx']
     snip : ['Core legal use cases']

[4] (easy) What are the five evaluation criteria used for SignChain AI?
     gold : ['data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx']
     snip : ['Depth of exploration\nPractical thinking\nAbility to identify tr

In [46]:
import unicodedata


def _normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"\s+", " ", text).strip()


def snippet_in_docs(snippet: str, docs: list[Document], min_overlap: float = 0.6) -> bool:
    """Gold snippet present in any retrieved doc: exact (normalized) OR token-overlap."""
    if not snippet:
        return False
    nsnip = _normalize(snippet)
    for d in docs:
        if nsnip in _normalize(d.page_content):
            return True
    snip_tokens = set(nsnip.split())
    if not snip_tokens:
        return False
    for d in docs:
        doc_tokens = set(_normalize(d.page_content).split())
        if len(snip_tokens & doc_tokens) / len(snip_tokens) >= min_overlap:
            return True
    return False


def evaluate_retriever(retrieve_fn, eval_set: list[EvalExample], k: int = 5) -> dict:
    """Hit@k, Recall@k, MRR, Snippet-Hit@k for a single retriever."""
    n = hits = snippet_hits = 0
    recall_sum = rr_sum = 0.0
    per_diff = defaultdict(lambda: {"n": 0, "hit": 0})

    for ex in eval_set:
        gold = set(ex.gold_source_paths)
        if not gold:
            continue  # negatives belong to the generation/threshold eval
        n += 1

        docs = retrieve_fn(ex.question, k)
        sources = [d.metadata.get("source_path") for d in docs]

        hit = any(s in gold for s in sources)
        hits += int(hit)

        recall_sum += len({s for s in sources if s in gold}) / len(gold)

        rr = 0.0
        for rank, s in enumerate(sources, 1):
            if s in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr

        if any(snippet_in_docs(sn, docs) for sn in ex.gold_snippets):
            snippet_hits += 1

        per_diff[ex.difficulty]["n"] += 1
        per_diff[ex.difficulty]["hit"] += int(hit)

    if n == 0:
        return {"n": 0}

    return {
        "n": n,
        "hit@k": hits / n,
        "recall@k": recall_sum / n,
        "mrr": rr_sum / n,
        "snippet_hit@k": snippet_hits / n,
        "by_difficulty": {
            d: round(v["hit"] / v["n"], 3)
            for d, v in sorted(per_diff.items()) if v["n"]
        },
    }

In [49]:
# Uniform adapters: (query, k) -> list[Document]
RETRIEVERS = {
    "dense":           lambda q, k: [d for d, _ in vectorstore.similarity_search_with_score(q, k=k)],
    "bm25":            lambda q, k: [d for d, _ in bm25_retriever.retrieve(q, k=k)],
    "hybrid":          lambda q, k: [d for d, _ in ensemble.retrieve(q, top_k=k)],
    "hybrid_reranked": lambda q, k: [d for d, _ in hybrid_reranked.retrieve(q, top_k=k)],
}

EVAL_K = cfg.TOP_K
n_pos = len([e for e in eval_set if e.gold_source_paths])
print(f"Evaluating {n_pos} positive examples @ k={EVAL_K}\n")

results = {}
for name, fn in RETRIEVERS.items():
    log.info(f"Evaluating: {name}")
    results[name] = evaluate_retriever(fn, eval_set, k=EVAL_K)

print(f"\n{'Retriever':<18}{'Hit@k':>8}{'Recall@k':>10}{'MRR':>8}{'Snippet@k':>11}")
print("-" * 55)
for name, r in results.items():
    if r.get("n", 0) == 0:
        continue
    print(f"{name:<18}{r['hit@k']:>8.3f}{r['recall@k']:>10.3f}"
          f"{r['mrr']:>8.3f}{r['snippet_hit@k']:>11.3f}")
print("-" * 55)

print("\nHit@k by difficulty:")
for name, r in results.items():
    if r.get("n", 0):
        print(f"  {name:<18} {r['by_difficulty']}")

# Persist for regression history
RESULTS_DIR = EVAL_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
run_path = RESULTS_DIR / f"retrieval_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.json"
run_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
log.info(f"Saved run → {run_path.relative_to(PROJECT_ROOT)}")

2026-05-18 16:55:46,535 - INFO    | rag | Evaluating: dense


Evaluating 23 positive examples @ k=5



2026-05-18 16:55:47,599 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:47,673 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:47,747 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:47,818 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:47,886 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:47,957 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:48,030 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:48,103 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 16:55:48,169 - INFO    | httpx | HTTP Request: POST http://localhost:11434/ap


Retriever            Hit@k  Recall@k     MRR  Snippet@k
-------------------------------------------------------
dense                1.000     1.000   0.946      0.957
bm25                 1.000     1.000   0.878      0.870
hybrid               1.000     1.000   0.978      1.000
hybrid_reranked      1.000     1.000   0.978      1.000
-------------------------------------------------------

Hit@k by difficulty:
  dense              {'easy': 1.0, 'hard': 1.0, 'medium': 1.0}
  bm25               {'easy': 1.0, 'hard': 1.0, 'medium': 1.0}
  hybrid             {'easy': 1.0, 'hard': 1.0, 'medium': 1.0}
  hybrid_reranked    {'easy': 1.0, 'hard': 1.0, 'medium': 1.0}


In [48]:
NEW_HARD_CASES = [
    # ---------- NEGATIVES (system SHOULD refuse) ----------
    {
        "question": "What is the company's parental leave policy?",
        "gold_source_paths": [],
        "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "easy",
        "notes": "NEGATIVE: HR policy absent from corpus (low lexical overlap)"
    },
    {
        "question": "What is the salary range offered for the data scientist role?",
        "gold_source_paths": [],
        "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "easy",
        "notes": "NEGATIVE: compensation absent (low lexical overlap)"
    },
    {
        "question": "How many engineers are on the legal AI team?",
        "gold_source_paths": [],
        "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "medium",
        "notes": "NEGATIVE: team size absent (medium overlap on 'legal AI')"
    },
    {
        "question": "Which vector database did the team benchmark for production latency?",
        "gold_source_paths": [],
        "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "hard",
        "notes": "ADVERSARIAL NEGATIVE: in-domain, dense will pull RAG/vector chunks; no benchmark exists. Tests min_score / hallucination."
    },
    {
        "question": "What were the measured results of the SignChain AI prototype evaluation?",
        "gold_source_paths": [],
        "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "hard",
        "notes": "ADVERSARIAL NEGATIVE: high overlap with 'SignChain AI' + 'evaluation criteria'; results don't exist (task not done)."
    },
    {
        "question": "What is the penalty for late submission of the Stage 1 deliverable?",
        "gold_source_paths": [],
        "gold_snippets": [],
        "reference_answer": "I don't have that information in the provided documents.",
        "difficulty": "hard",
        "notes": "ADVERSARIAL NEGATIVE: 'Stage 1 deliverable' overlap; penalty terms absent."
    },

    # ---------- PARAPHRASES (same answer, keyword-stripped) ----------
    {
        "question": "Which recreational competitions did he run for residents at his student accommodation?",
        "gold_source_paths": ["data/raw/pdfs/CV_Himanshu.pdf"],
        "gold_snippets": ["Cricket, Chess, Tennis"],
        "reference_answer": None,
        "difficulty": "medium",
        "notes": "PARAPHRASE of 'sporting events at Unilodge' — no shared keywords with source"
    },
    {
        "question": "Where was the supercomputer he ran his cosmological code on located?",
        "gold_source_paths": ["data/raw/pdfs/CV_Himanshu.pdf"],
        "gold_snippets": ["Indian Institute of Technology Kharagpur (IITKGP)"],
        "reference_answer": None,
        "difficulty": "hard",
        "notes": "PARAPHRASE — strips the distinctive 'Shakti'/'HPC' keywords BM25 relied on; pure dense test"
    },
    {
        "question": "What are the core focus areas for developing a trustworthy AI system in the legal domain?",
        "gold_source_paths": ["data/raw/docx/Stage 1_ AI Onboarding - Himanshu.docx"],
        "gold_snippets": ["Context & Retrieval Systems (RAG evolved), Structured AI Outputs & Intelligence Layer, and Agentic Workflows (multi-step reasoning systems)"],
        "reference_answer": None,
        "difficulty": "medium",
        "notes": "PARAPHRASE of 'three pillars for reliable Legal AI'"
    },
    {
        "question": "What software-design abilities are required to build a cleanly separated, easy-to-extend data flow for an LLM retrieval system?",
        "gold_source_paths": ["data/raw/docx/Self-Assessment - Himanshu T.docx"],
        "gold_snippets": ["Pipeline modularization\nSeparation of concerns\nMaintainability and testability"],
        "reference_answer": None,
        "difficulty": "medium",
        "notes": "PARAPHRASE of 'competencies for designing a modular pipeline'"
    },
]

existing = load_eval_set(EVAL_SET_PATH)
seen = {e.question.strip().lower() for e in existing}
added = 0
for d in NEW_HARD_CASES:
    if d["question"].strip().lower() in seen:
        continue
    existing.append(EvalExample(**d))
    added += 1

save_eval_set(existing, EVAL_SET_PATH)
eval_set = existing
log.info(f"Added {added} hard cases → {len(eval_set)} examples "
         f"({len([e for e in eval_set if not e.gold_source_paths])} negatives)")

2026-05-18 16:55:15,141 - INFO    | rag | Saved 29 examples → eval/eval_set.json
2026-05-18 16:55:15,141 - INFO    | rag | Added 10 hard cases → 29 examples (6 negatives)


## 
###### 2: Aspects of RAG (testing RAG system)
###### 3: experiments with RAG:
###### top 10:
###### learning UI/integration
###### multiple RAG 
###### think of ways to implement multiple embedding models, and LLM models
##